#### Factory Method Pattern

Imagine we have an e-commerce platform. For MVP, we integrate `Stripe` to handle credit cards. And our checkout logic creates a Stripe object directly like ...
```python
 gateway = new StripeAPI()
```
And after 6 months, we expands our business and product team wants to integrate PayPal and later crypto payments. But if our checkout system is hardcoded to use Stripe, we'll have to update our checkout logic to add a massive `if/else` statements.

*The Code Smell*: Directly instantiating concrete classes binds your high-level business logic (the checkout process) to low-level implementation details (specific API wrappers). This violates the Open/Closed Principle.

#### The Solution

The Factory Method pattern replaces direct object construction calls with calls to a special factory method to construct objects.

Instead of the checkout system creating the Stripe API directly, we delegate the creation to a dedicated Creator class. The base Creator class declares the factory method but leaves it up to its subclasses to actually implement it and decide which concrete payment gateway to instantiate.

**Important Note**: The factory method in the base class must declare its return type as a common Interface or Base Class. As long as all products implement this interface, the client code can treat them uniformly, completely ignorant of the concrete types.

**The Components**
To implement this, we need the following:
- `Product (Interface)`: Defines the common contract that all objects created by the factory must follow.
- `Concrete Product`: Actual implementation of Product interface (e.g., `StripeGateway`, `PayPalGateway`)
- `Creator (Abstract)`: Declares the factory_method(). It contains core business logic (the checkout flow) that relies on the Product objects.
- `Concrete Creator`: Overrides the factory_method() to return a specific Concrete Product.

In [1]:
from abc import ABC, abstractmethod

# ==========================================
# 1. Product Hierarchy (The Interfaces)
# ==========================================
class PaymentGateway(ABC):
    """The common interface for all payment providers."""
    @abstractmethod
    def charge(self, amount: float) -> str:
        pass

class StripeGateway(PaymentGateway):
    def charge(self, amount: float) -> str:
        # Complex Stripe API logic would go here
        return f"[Stripe] Successfully charged ${amount} to credit card."

class PayPalGateway(PaymentGateway):
    def charge(self, amount: float) -> str:
        # Complex PayPal API logic would go here
        return f"[PayPal] Successfully deducted ${amount} from user wallet."

# ==========================================
# 2. Creator Hierarchy (The Factories)
# ==========================================
class PaymentProcessor(ABC):
    """
    The Creator class declares the factory method. 
    It also contains the core business logic that uses the product!
    """
    @abstractmethod
    def create_gateway(self) -> PaymentGateway:
        """The Factory Method"""
        pass

    def process_checkout(self, amount: float) -> str:
        """Core business logic that relies on the factory method."""
        # 1. Create the product via the factory method
        gateway = self.create_gateway()
        
        # 2. Use the product via its abstract interface
        # We can add universal pre/post processing here (logging, taxes, etc.)
        print(f"System: Initiating secure checkout for ${amount}...")
        result = gateway.charge(amount)
        
        return result

class StripeProcessor(PaymentProcessor):
    def create_gateway(self) -> PaymentGateway:
        return StripeGateway()

class PayPalProcessor(PaymentProcessor):
    def create_gateway(self) -> PaymentGateway:
        return PayPalGateway()

# ==========================================
# 3. Client Perspective
# ==========================================
def client_checkout(processor: PaymentProcessor, amount: float):
    """
    The client doesn't know WHICH processor it was given, 
    nor does it know WHICH gateway is being created.
    """
    receipt = processor.process_checkout(amount)
    print(receipt)

# --- Usage ---
print("--- User selects Credit Card ---")
cc_system = StripeProcessor()
client_checkout(cc_system, 150.00)

print("\n--- User selects PayPal ---")
paypal_system = PayPalProcessor()
client_checkout(paypal_system, 45.50)

--- User selects Credit Card ---
System: Initiating secure checkout for $150.0...
[Stripe] Successfully charged $150.0 to credit card.

--- User selects PayPal ---
System: Initiating secure checkout for $45.5...
[PayPal] Successfully deducted $45.5 from user wallet.


**Execution Trace**

- first client calls the `cc_system = StripeProcessor()`.
- then it is passed into `client_checkout`, which calls `cc_system.process_checkout(150.00)`.
- **Inheritance Kicks In**: Python looks at `StripeProcessor` and says, "Do you have a method called `process_checkout`?" The answer is **no**. But because it inherits from `PaymentProcessor`, Python climbs up the inheritance tree and finds the method in the base class. It starts executing that base class code.
- **Dynamic Dispatch (The Core Concept)**: Inside `process_checkout`, Python hits `gateway = self.create_gateway()`.
- **The Context of `self`**: Even though we are running code written inside the base `PaymentProcessor` class, `self` is still pointing to the `StripeProcessor` object we created in Step 1.
- **Method Resolution**: Therefore, Python looks for `create_gateway()` inside `StripeProcessor`. It finds it, runs it, and returns a fresh `StripeGateway` object.
- **Using the Product**: Now `gateway` holds a `StripeGateway`. When the code hits `gateway.charge(amount)`, it executes the **Stripe-specific logic**.

**Why this is better**:
- **Decoupling**: Your core e-commerce logic is entirely decoupled from external vendor APIs.
- **Single Responsibility Principle**: The code that handles the messy API setup for Stripe is isolated in the StripeProcessor and StripeGateway.
- **Open/Closed Principle**: Next year, when the CTO asks for Apple Pay, you just create ApplePayGateway and ApplePayProcessor. You do not touch the existing PaymentProcessor or client code. The core system remains stable and untouched.